# Avaliação walk-forward:XGBoost

Walk-forward é uma técnica de validação cruzada usada em séries temporais, onde o modelo é treinado em um conjunto de dados e testado em um conjunto subsequente. Isso é feito repetidamente, movendo a janela de treinamento e teste para frente no tempo.


In [6]:
!pip install -U catboost xgboost scikit-learn pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 25.9 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 44.2 MB/s  0:00:01m0:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1━━━ 0/3 [pyarrow]
  Attempting uninstall: scikit-learn━━━━━━━━━━━━ 0/3 [pyarrow]
    Found existing installation: scikit-learn 1.6.1━━━━━━━━━━━━━━━ 1/3 [scikit-learn]
    Uninstalling scikit-learn-1.6.1:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [scikit-learn]
      Successfully uninstalled scikit-learn-1.6.1━━━━━━━━━━━━━━━━━ 1/3 [scikit-learn]
  Attempting uninstall: pandas╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [scikit-learn]
    Found existing installation: pandas 2.3.3━━━━━━━━━━━━━━━━━ 1/3 [scikit-learn]
    Uninstalling pandas-2.3.3:━━━━╸━━━━━━━━━━━━━ 2/3 [pandas]
      Successfully uninstalled pand

In [7]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_score, recall_score, f1_score)

In [8]:
import os
pathKaggle = "../input/datasets/luisfilipelima/bancotransacao/"
path = "../../data/processed/"
pathDrive = "/content/drive/MyDrive/TreinosIAS/XGBoosting/dados/"
#path=pathDrive
path = pathKaggle
path

'../input/datasets/luisfilipelima/bancotransacao/'

In [9]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

obt = pd.read_parquet(path + "obt_with_terminal_history.parquet")
obtTrain = pd.read_parquet(path + "obt_train.parquet")
obtTest = pd.read_parquet(path + "obt_validation.parquet")

obtTrain.head()

Pandas: 2.3.3
PyArrow: 23.0.1


,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_category,hour,day_week,terminal_age_days,is_fraud,...,terminal_tx_count_24h,terminal_distinct_cards_24h,seconds_since_last_terminal_tx,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d,card_distinct_terminals,terminal_fraud_rate,card_fraud_rate
0,1156634,2018-02-15 13:34:00,12.400000,3520842735991929,847,SmartTalk Communications,13,4,173.0,False,...,24,23,1396.0,165,12.378606,8.682511,0.002464,155,0.001805,0.003953
1,1157370,2018-02-15 13:57:03,52.340000,370934368869147,847,SmartTalk Communications,13,4,173.0,False,...,25,24,1383.0,123,105.470894,91.224616,-0.582418,134,0.001803,0.000000
2,1157375,2018-02-15 13:57:17,78.169998,6011689217447071,847,SmartTalk Communications,13,4,173.0,False,...,26,25,14.0,122,139.657131,132.426997,-0.464310,168,0.001802,0.000000
3,1159536,2018-02-15 15:09:34,23.850000,2662393784087193,847,SmartTalk Communications,15,4,173.0,False,...,24,23,4337.0,142,91.744367,119.426299,-0.568504,149,0.001800,0.000000
4,1160251,2018-02-15 15:35:49,29.459999,3515828436568584,847,SmartTalk Communications,15,4,173.0,False,...,24,23,1575.0,102,43.886078,27.739705,-0.520052,113,0.001799,0.016575


In [10]:
cat_features = ["terminal_category"]

features = [
    "tx_amount",
    "hour",
    "day_week",
    "terminal_age_days",
    "terminal_fraud_rate",
    "card_fraud_rate",
    "card_distinct_terminals",
    "terminal_new_for_card",
    "card_tx_count_1h",
    "card_tx_count_24h",
    "card_distinct_terminals_24h",
    "seconds_since_last_card_tx",
    "terminal_tx_count_1h",
    "terminal_tx_count_24h",
    "terminal_distinct_cards_24h",
    "seconds_since_last_terminal_tx",
    "card_history_count_30d",
    "card_amount_mean_30d",
    "card_amount_std_30d",
    "amount_z_score_30d",
    "terminal_category",
]
target = "is_fraud"
def prepararParaXGBoostging(df):
    df["datetime"] = pd.to_datetime(df["tx_datetime"], errors="raise")
    df[target] = df[target].astype(int)

    # XGB possui um suporte nativo para dados categoricos, tem que definir o tipo do dado como "category"
    df["terminal_category"] = (
        df["terminal_category"]
        .astype("object")
        .fillna("SEM_CATEGORIA")
        .astype("category")
)
    df = df.sort_values("tx_datetime").reset_index(drop=True)
    return df


    
obt = prepararParaXGBoostging(obt)
obtTrain = prepararParaXGBoostging(obtTrain)
obtTest = prepararParaXGBoostging(obtTest)


In [11]:
from xgboost import XGBClassifier

In [12]:
def calcular_lucro(TransacoesAceitas):
    if TransacoesAceitas.empty:
        print("Nenhuma transação aceita; lucro = 0.")
        return 0.0
    legitimas = TransacoesAceitas["is_fraud"].eq(0)
    fraudulentas = TransacoesAceitas["is_fraud"].eq(1)

    receita = (
        0.03 * TransacoesAceitas.loc[legitimas, "tx_amount"].sum()
    )

    prejuizo = (
        TransacoesAceitas.loc[fraudulentas, "tx_amount"].sum()
    )
    print("CALCULANDO MELHOR LUCRO...")
    print(f"VALOR:{float(receita - prejuizo)}")
    # Recusas têm resultado financeiro direto igual a zero.
    return float(receita - prejuizo).__round__(2)


def classificarTransacoes(x,y,prob,limiar):
    transacoes = x.copy()
    transacoes["is_fraud"] = np.asarray(y, dtype=int)
    transacoes["probabilidade_fraude"] = prob

    aceita = transacoes["probabilidade_fraude"] < limiar

    TransacoesAceitas = transacoes.loc[aceita].copy()
    TransacoesRecusadas = transacoes.loc[~aceita].copy() # pega todas menos as acceitas

    return TransacoesAceitas, TransacoesRecusadas

def encontrar_melhor_limiar(modelo, X_val, y_val):
    probabilidades = modelo.predict_proba(X_val)[:, 1]
    valores = X_val["tx_amount"].to_numpy()
    y = np.asarray(y_val, dtype=int)

    # Ordena da menor para a maior chance de fraude. Assim cada posição
    # representa o lucro acumulado ao aceitar as transações até aquele ponto.
    ordem = np.argsort(probabilidades, kind="stable")
    probabilidades_ordenadas = probabilidades[ordem]
    valores_ordenados = valores[ordem]
    y_ordenado = y[ordem]

    lucro_por_transacao = np.where(
        y_ordenado == 0,
        0.03 * valores_ordenados,
        -valores_ordenados,
    )
    lucro_acumulado = np.cumsum(lucro_por_transacao)

    
    fim_de_grupo = np.concatenate([
        probabilidades_ordenadas[1:] != probabilidades_ordenadas[:-1],
        [True],
    ])
    indices = np.flatnonzero(fim_de_grupo)
    lucros = lucro_acumulado[indices]

    
    lucros = np.concatenate([[0.0], lucros])
    melhor_posicao = int(np.argmax(lucros))
    melhor_lucro = float(lucros[melhor_posicao])

    if melhor_posicao == 0:
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[0], -np.inf
        )
    else:
        indice = indices[melhor_posicao - 1]
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[indice], np.inf
        )

    return float(melhor_limiar), melhor_lucro


In [13]:
from sklearn.model_selection import TimeSeriesSplit
import optuna
tscv = TimeSeriesSplit(n_splits=10)


In [14]:
SEED = 42
N_INIT_TRIALS = 10
N_TRIALS = 30
N_SPLITS = 10


Buscatrain = obtTrain.tail(300_000).copy()
X_Train = Buscatrain[features].copy().reset_index(drop=True)
y_Train = Buscatrain[target].copy().reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

fixed_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "random_state": SEED,
    "n_jobs": 4,
    "verbosity": 0,
}

def preparar_fold_xgb(X_fit, X_val):
    """Alinha categorias sem usar rótulos nem alterar a ordem temporal."""
    X_fit = X_fit.copy()
    X_val = X_val.copy()
    for coluna in cat_features:
        categorias = pd.Index(
            pd.concat([X_fit[coluna], X_val[coluna]], ignore_index=True).unique()
        )
        X_fit[coluna] = pd.Categorical(X_fit[coluna], categories=categorias)
        X_val[coluna] = pd.Categorical(X_val[coluna], categories=categorias)
    return X_fit, X_val

def objetivo(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 20.0, log=True),
    }

    lucros_folds = []
    prauc_folds = []
    limiares_folds = []

    for train_idx, val_idx in tscv.split(X_Train):
        X_fit, X_val = X_Train.iloc[train_idx], X_Train.iloc[val_idx]
        y_fit, y_val = y_Train.iloc[train_idx], y_Train.iloc[val_idx]
        X_fit, X_val = preparar_fold_xgb(X_fit, X_val)

        model = XGBClassifier(**fixed_params, **params)
        model.fit(X_fit, y_fit, verbose=False)

        probabilidades = model.predict_proba(X_val)[:, 1]
        # Cada limiar é escolhido somente dentro do fold de validação correspondente.
        limiar_fold, lucro_fold = encontrar_melhor_limiar(
            model, X_val, y_val
        )
        lucros_folds.append(lucro_fold)
        limiares_folds.append(limiar_fold)
        prauc_folds.append(average_precision_score(y_val, probabilidades))

    trial.set_user_attr("lucros_por_fold", lucros_folds)
    trial.set_user_attr("limiares_por_fold", limiares_folds)
    trial.set_user_attr("prauc_media", float(np.mean(prauc_folds)))
    return float(np.mean(lucros_folds))

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=N_INIT_TRIALS,
    ),
)
study.optimize(
    objetivo,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)




[I 2026-11-10 22:50:00,022] A new study created in memory with name: no-name-c7e66896-5ba5-4e3e-a86f-87ad7992353d


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-10 22:50:30,152] Trial 0 finished with value: 50392.303125 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'min_child_weight': 8.960785365368121, 'gamma': 2.993292420985183, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 21.42302175774105, 'scale_pos_weight': 6.054365855469246}. Best is trial 0 with value: 50392.303125.
[I 2026-11-10 22:51:17,872] Trial 1 finished with value: 50440.371484375 and parameters: {'max_depth': 7, 'learning_rate': 0.010636066512540286, 'min_child_weight': 18.276027831785726, 'gamma': 4.162213204002109, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 0.033205591037519584, 'scale_pos_weight': 4.816414530907083}. Best is trial 1 with value: 50440.371484375.
[I 2026-11-10 22:51:59,920] Trial 2 finished with value: 50413.55 and parameters: {'max_depth': 6, 'learning_rate': 0.023927

## Resultados

In [15]:
print("Lucro médio dos folds da melhor configuração:", study.best_value)
print("Melhores hiperparâmetros:", study.best_params)
print("PRAUC médio nos folds:", study.best_trial.user_attrs["prauc_media"])
print("Lucros por fold:", study.best_trial.user_attrs["lucros_por_fold"])
print("Limiares por fold:", study.best_trial.user_attrs["limiares_por_fold"])

Lucro médio dos folds da melhor configuração: 50512.535546875
Melhores hiperparâmetros: {'max_depth': 5, 'learning_rate': 0.02465472058004036, 'min_child_weight': 3.7601042135007705, 'gamma': 0.1999351487862493, 'subsample': 0.9902173701155406, 'colsample_bytree': 0.9808595220254002, 'reg_alpha': 7.749107484746391, 'reg_lambda': 51.33972250700968, 'scale_pos_weight': 1.542072513132659}
PRAUC médio nos folds: 0.5101366833551108
Lucros por fold: [51065.73046875, 51566.6875, 49573.94140625, 48897.03515625, 50171.80859375, 48460.0703125, 52071.20703125, 49532.3125, 52857.3359375, 50929.2265625]
Limiares por fold: [0.62343829870224, 0.34031933546066284, 0.3107077181339264, 0.20919270813465118, 0.048076946288347244, 0.043765030801296234, 0.14498496055603027, 0.10566635429859161, 0.0695778876543045, 0.02272547408938408]


In [18]:
import xgboost  as xgb
x_test, y_test = obtTest[features].copy(), obtTest[target].copy()
melhores_Parametros = study.best_params
#print(study.best_params.user_attrs["n_trees"])
modelo_final = xgb.XGBClassifier(
    **fixed_params,
    **melhores_Parametros
   # n_estimators=study.best_trial.user_attrs["n_trees"],
)

modelo_final.fit(
    Buscatrain[features],
    Buscatrain[target],
    verbose=False,
)


limiar_final, lucro_limiar = encontrar_melhor_limiar(
    modelo_final,
    x_test,
    y_test,
)

print("Melhor limiar:", limiar_final)
print("Lucro:", lucro_limiar)

Melhor limiar: 0.04948967695236206
Lucro: 1809581.625


In [19]:


probabilidades_teste = modelo_final.predict_proba(x_test)[:, 1]
predicoes_teste = (probabilidades_teste >= limiar_final).astype(int)

resultado_teste = x_test.copy()

resultado_teste["probabilidade_fraude"] = probabilidades_teste
resultado_teste["fraude_prevista"] = predicoes_teste
resultado_teste["fraude_real"] = y_test.to_numpy()

TransacoesAceitas, TransacoesRecusadas = classificarTransacoes(
    x_test,
    y_test,
    probabilidades_teste,
    limiar_final,
)

lucro_teste = calcular_lucro(TransacoesAceitas)
matriz = confusion_matrix(y_test, predicoes_teste)
precisao = precision_score(y_test, predicoes_teste, zero_division=0)
recall = recall_score(y_test, predicoes_teste, zero_division=0)
prauc_teste = average_precision_score(y_test, probabilidades_teste)

print("Matriz de confusão:")
print(matriz)
print("Lucro em maio:", lucro_teste)
print("Taxa de transações recusadas:", len(TransacoesRecusadas) / len(x_test))

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 0)
]
print("erro / total")
print(f"{len(res)}/{len(resultado_teste.loc[resultado_teste["fraude_real"] == 1])}")

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 1)
]
res
print("certo / total")
print(f"{len(res)}/{len(resultado_teste.loc[resultado_teste["fraude_real"] == 1])}")


CALCULANDO MELHOR LUCRO...
VALOR:1809628.625
Matriz de confusão:
[[1039536    2282]
 [   7252    5982]]
Lucro em maio: 1809628.62
Taxa de transações recusadas: 0.007832789284319635
erro / total
7252/13234
certo / total
5982/13234


In [21]:
for parametro, valor in melhores_Parametros.items():
    print(f"{parametro}: {valor}")

max_depth: 5
learning_rate: 0.02465472058004036
min_child_weight: 3.7601042135007705
gamma: 0.1999351487862493
subsample: 0.9902173701155406
colsample_bytree: 0.9808595220254002
reg_alpha: 7.749107484746391
reg_lambda: 51.33972250700968
scale_pos_weight: 1.542072513132659
